# Spatial Transcriptomics Neighborhood Analysis in Tumor Tissue

As a bioinformatics scientist, I am using a fully synthetic spatial transcriptomics dataset to ask a tumor microenvironment question: do tumor-rich regions exclude immune cells, or do immune cells infiltrate specific neighborhoods near the invasive margin?

The study project is `ST-TME-BENCHMARK-042`, with two synthetic samples (`STTB-001` and `STTB-002`) and eight spatial neighborhoods spanning marker names, project IDs, sample regions, metric values, and conclusions.

In [2]:
from collections import Counter, defaultdict
import random

PROJECT_ID = "ST-TME-BENCHMARK-042"
SAMPLE_IDS = ["STTB-001", "STTB-002"]
RANDOM_SEED = 1729
MARKERS = ["EPCAM", "COL1A1", "CD8A", "PDCD1", "CXCL13"]
rng = random.Random(RANDOM_SEED)

print("project_id", PROJECT_ID)
print("sample_ids", ", ".join(SAMPLE_IDS))
print("random_seed", RANDOM_SEED)
print("marker_panel", ", ".join(MARKERS))

project_id ST-TME-BENCHMARK-042
sample_ids STTB-001, STTB-002
random_seed 1729
marker_panel EPCAM, COL1A1, CD8A, PDCD1, CXCL13


## Marker Interpretation

I use `EPCAM` as an epithelial tumor marker and `COL1A1` as a stromal fibroblast or collagen-rich matrix marker. `CD8A` and `PDCD1` are immune markers: `CD8A` marks cytotoxic T cell infiltration, while `PDCD1` marks activated or exhausted immune cells that may reflect checkpoint-associated T cell states. `CXCL13` marks immune niche organization, including tertiary lymphoid-like recruitment programs.

In [3]:
def format_table(rows, columns):
    widths = {column: max(len(column), *(len(str(row[column])) for row in rows)) for column in columns}
    lines = ["  ".join(column.ljust(widths[column]) for column in columns)]
    for row in rows:
        lines.append("  ".join(str(row[column]).ljust(widths[column]) for column in columns))
    return "\n".join(lines)

neighborhood_plan = [
    {"neighborhood_id": "NBR_TME_01", "sample_id": "STTB-001", "region": "tumor_core", "center_x": 18.0, "center_y": 22.0, "planned_program": "immune_excluded"},
    {"neighborhood_id": "NBR_TME_02", "sample_id": "STTB-001", "region": "tumor_core", "center_x": 32.0, "center_y": 25.0, "planned_program": "immune_excluded"},
    {"neighborhood_id": "NBR_TME_03", "sample_id": "STTB-001", "region": "stromal_rim", "center_x": 50.0, "center_y": 33.0, "planned_program": "immune_excluded"},
    {"neighborhood_id": "NBR_TME_04", "sample_id": "STTB-002", "region": "tumor_core", "center_x": 21.0, "center_y": 54.0, "planned_program": "immune_excluded"},
    {"neighborhood_id": "NBR_TME_05", "sample_id": "STTB-002", "region": "fibrotic_edge", "center_x": 43.0, "center_y": 58.0, "planned_program": "immune_excluded"},
    {"neighborhood_id": "NBR_TME_06", "sample_id": "STTB-001", "region": "invasive_margin", "center_x": 63.0, "center_y": 21.0, "planned_program": "immune_infiltrated"},
    {"neighborhood_id": "NBR_TME_07", "sample_id": "STTB-002", "region": "tertiary_lymphoid_like", "center_x": 70.0, "center_y": 47.0, "planned_program": "immune_infiltrated"},
    {"neighborhood_id": "NBR_TME_08", "sample_id": "STTB-002", "region": "invasive_margin", "center_x": 59.0, "center_y": 69.0, "planned_program": "immune_infiltrated"},
]

print(format_table(neighborhood_plan, ["neighborhood_id", "sample_id", "region", "center_x", "center_y", "planned_program"]))

neighborhood_id  sample_id  region                  center_x  center_y  planned_program   
NBR_TME_01       STTB-001   tumor_core              18.0      22.0      immune_excluded   
NBR_TME_02       STTB-001   tumor_core              32.0      25.0      immune_excluded   
NBR_TME_03       STTB-001   stromal_rim             50.0      33.0      immune_excluded   
NBR_TME_04       STTB-002   tumor_core              21.0      54.0      immune_excluded   
NBR_TME_05       STTB-002   fibrotic_edge           43.0      58.0      immune_excluded   
NBR_TME_06       STTB-001   invasive_margin         63.0      21.0      immune_infiltrated
NBR_TME_07       STTB-002   tertiary_lymphoid_like  70.0      47.0      immune_infiltrated
NBR_TME_08       STTB-002   invasive_margin         59.0      69.0      immune_infiltrated


In [4]:
marker_profiles = {
    "tumor_epithelial": {"EPCAM": 6.2, "COL1A1": 0.7, "CD8A": 0.2, "PDCD1": 0.2, "CXCL13": 0.2},
    "stromal_fibroblast": {"EPCAM": 0.8, "COL1A1": 5.8, "CD8A": 0.2, "PDCD1": 0.2, "CXCL13": 0.3},
    "cd8_tcell": {"EPCAM": 0.4, "COL1A1": 0.6, "CD8A": 5.5, "PDCD1": 1.4, "CXCL13": 1.1},
    "pdcd1_cd8_tcell": {"EPCAM": 0.4, "COL1A1": 0.6, "CD8A": 5.0, "PDCD1": 4.8, "CXCL13": 1.3},
    "cxcl13_immune_niche": {"EPCAM": 0.5, "COL1A1": 0.7, "CD8A": 3.5, "PDCD1": 2.3, "CXCL13": 5.2},
}

phenotype_templates = {
    "immune_excluded": ["tumor_epithelial"] * 9 + ["stromal_fibroblast"] * 8 + ["cd8_tcell", "pdcd1_cd8_tcell", "cxcl13_immune_niche"],
    "immune_infiltrated": ["tumor_epithelial"] * 6 + ["stromal_fibroblast"] * 3 + ["cd8_tcell"] * 5 + ["pdcd1_cd8_tcell"] * 3 + ["cxcl13_immune_niche"] * 3,
}

spots = []
for neighborhood in neighborhood_plan:
    template = phenotype_templates[neighborhood["planned_program"]]
    for spot_index, phenotype_seed in enumerate(template, start=1):
        marker_values = {
            marker: round(max(0.0, marker_profiles[phenotype_seed][marker] + rng.gauss(0, 0.05)), 2)
            for marker in MARKERS
        }
        spot = {
            "spot_id": f"{neighborhood['neighborhood_id']}_SPOT_{spot_index:02d}",
            "sample_id": neighborhood["sample_id"],
            "neighborhood_id": neighborhood["neighborhood_id"],
            "region": neighborhood["region"],
            "x": round(neighborhood["center_x"] + rng.gauss(0, 2.5), 2),
            "y": round(neighborhood["center_y"] + rng.gauss(0, 2.5), 2),
            "phenotype_seed": phenotype_seed,
        }
        spot.update(marker_values)
        spots.append(spot)

print("synthetic_spot_count", len(spots))
print(format_table(spots[:6], ["spot_id", "sample_id", "region", "x", "y", "EPCAM", "COL1A1", "CD8A", "PDCD1", "CXCL13"]))

synthetic_spot_count 160
spot_id             sample_id  region      x      y      EPCAM  COL1A1  CD8A  PDCD1  CXCL13
NBR_TME_01_SPOT_01  STTB-001   tumor_core  16.62  21.38  6.3    0.7     0.15  0.23   0.18  
NBR_TME_01_SPOT_02  STTB-001   tumor_core  17.65  19.08  6.25   0.72    0.17  0.21   0.17  
NBR_TME_01_SPOT_03  STTB-001   tumor_core  14.52  19.58  6.16   0.67    0.15  0.26   0.2   
NBR_TME_01_SPOT_04  STTB-001   tumor_core  24.45  19.5   6.18   0.72    0.21  0.25   0.16  
NBR_TME_01_SPOT_05  STTB-001   tumor_core  17.49  25.08  6.29   0.65    0.11  0.12   0.25  
NBR_TME_01_SPOT_06  STTB-001   tumor_core  15.29  22.34  6.18   0.58    0.14  0.08   0.23  


In [5]:
marker_summary = []
for marker in MARKERS:
    values = [spot[marker] for spot in spots]
    marker_summary.append({
        "marker": marker,
        "mean": f"{sum(values) / len(values):.2f}",
        "max": f"{max(values):.2f}",
    })

print(format_table(marker_summary, ["marker", "mean", "max"]))

marker  mean  max 
EPCAM   2.81  6.34
COL1A1  2.24  5.90
CD8A    1.56  5.55
PDCD1   0.94  4.90
CXCL13  0.87  5.30


## Phenotype Assignment Strategy

Next I convert marker expression into spot phenotypes. This mirrors a simple rule-based first pass I would use before fitting heavier spatial models: high `EPCAM` assigns tumor epithelial spots, high `COL1A1` assigns stromal fibroblasts, and immune spots are split by `CD8A`, `PDCD1`, and `CXCL13` signal. Marker-based rules keep tumor, stromal, and immune compartments explicit in the spot labels.

In [6]:
def assign_spot_phenotype(spot):
    if spot["EPCAM"] >= max(spot["COL1A1"], spot["CD8A"], spot["PDCD1"], spot["CXCL13"]):
        return "tumor_epithelial"
    if spot["COL1A1"] >= max(spot["EPCAM"], spot["CD8A"], spot["PDCD1"], spot["CXCL13"]):
        return "stromal_fibroblast"
    if spot["CXCL13"] >= 4.0:
        return "cxcl13_immune_niche"
    if spot["PDCD1"] >= 3.5:
        return "pdcd1_cd8_tcell"
    return "cd8_tcell"

for spot in spots:
    spot["assigned_phenotype"] = assign_spot_phenotype(spot)

phenotype_counts = Counter(spot["assigned_phenotype"] for spot in spots)
phenotype_count_rows = [
    {"assigned_phenotype": phenotype, "spots": phenotype_counts[phenotype]}
    for phenotype in ["tumor_epithelial", "stromal_fibroblast", "cd8_tcell", "pdcd1_cd8_tcell", "cxcl13_immune_niche"]
]

print(format_table(phenotype_count_rows, ["assigned_phenotype", "spots"]))

assigned_phenotype   spots
tumor_epithelial     63   
stromal_fibroblast   49   
cd8_tcell            20   
pdcd1_cd8_tcell      14   
cxcl13_immune_niche  14   


In [7]:
def classify_spatial_neighborhoods(spot_table, neighborhood_column="neighborhood_id"):
    """Summarize each spatial neighborhood by compartment composition."""
    immune_labels = {"cd8_tcell", "pdcd1_cd8_tcell", "cxcl13_immune_niche"}
    grouped_spots = defaultdict(list)
    for spot in spot_table:
        grouped_spots[spot[neighborhood_column]].append(spot)

    rows = []
    for neighborhood_id in sorted(grouped_spots):
        group = grouped_spots[neighborhood_id]
        counts = Counter(spot["assigned_phenotype"] for spot in group)
        total = len(group)
        immune_fraction = sum(counts[label] for label in immune_labels) / total
        rows.append({
            "neighborhood_id": neighborhood_id,
            "sample_id": group[0]["sample_id"],
            "region": group[0]["region"],
            "immune_state": "immune_infiltrated" if immune_fraction >= 0.45 else "immune_excluded",
            "tumor_fraction": f"{counts['tumor_epithelial'] / total:.2f}",
            "stromal_fraction": f"{counts['stromal_fibroblast'] / total:.2f}",
            "cd8a_pdcd1_fraction": f"{(counts['cd8_tcell'] + counts['pdcd1_cd8_tcell']) / total:.2f}",
            "cxcl13_fraction": f"{counts['cxcl13_immune_niche'] / total:.2f}",
            "immune_fraction": f"{immune_fraction:.2f}",
        })
    return rows

neighborhood_composition = classify_spatial_neighborhoods(spots)
composition_columns = [
    "neighborhood_id", "sample_id", "region", "immune_state", "tumor_fraction",
    "stromal_fraction", "cd8a_pdcd1_fraction", "cxcl13_fraction", "immune_fraction",
]

print(format_table(neighborhood_composition, composition_columns))
print("immune_excluded_neighborhoods", sum(1 for row in neighborhood_composition if row["immune_state"] == "immune_excluded"))
print("immune_infiltrated_neighborhoods", sum(1 for row in neighborhood_composition if row["immune_state"] == "immune_infiltrated"))

neighborhood_id  sample_id  region                  immune_state        tumor_fraction  stromal_fraction  cd8a_pdcd1_fraction  cxcl13_fraction  immune_fraction
NBR_TME_01       STTB-001   tumor_core              immune_excluded     0.45            0.40              0.10                 0.05             0.15           
NBR_TME_02       STTB-001   tumor_core              immune_excluded     0.45            0.40              0.10                 0.05             0.15           
NBR_TME_03       STTB-001   stromal_rim             immune_excluded     0.45            0.40              0.10                 0.05             0.15           
NBR_TME_04       STTB-002   tumor_core              immune_excluded     0.45            0.40              0.10                 0.05             0.15           
NBR_TME_05       STTB-002   fibrotic_edge           immune_excluded     0.45            0.40              0.10                 0.05             0.15           
NBR_TME_06       STTB-001   invasive_mar

In [8]:
x_min = min(spot["x"] for spot in spots)
x_max = max(spot["x"] for spot in spots)
y_min = min(spot["y"] for spot in spots)
y_max = max(spot["y"] for spot in spots)

print(
    "SPATIAL_SCATTER_PLOT_PLACEHOLDER "
    f"project_id={PROJECT_ID} spots={len(spots)} "
    f"x_range={x_min:.2f}-{x_max:.2f} y_range={y_min:.2f}-{y_max:.2f} "
    "color_by=immune_state"
)
print(
    "analysis_conclusion Immune exclusion dominates five tumor core or fibrotic neighborhoods, "
    "while NBR_TME_06, NBR_TME_07, and NBR_TME_08 show CD8A/PDCD1/CXCL13-enriched immune infiltration."
)

SPATIAL_SCATTER_PLOT_PLACEHOLDER project_id=ST-TME-BENCHMARK-042 spots=160 x_range=14.17-74.64 y_range=16.80-73.35 color_by=immune_state
analysis_conclusion Immune exclusion dominates five tumor core or fibrotic neighborhoods, while NBR_TME_06, NBR_TME_07, and NBR_TME_08 show CD8A/PDCD1/CXCL13-enriched immune infiltration.
